# CQTNet: Training From Scratch on Indian Music

## Section 1: Introduction & Setup

### What is Cover Song Identification?

Cover song identification (CSI) is a Music Information Retrieval (MIR) task where we want a system that can:
- Take any recording of a song
- Identify which *original* song it is a cover/version of

This is challenging because covers can differ in:
- **Key/pitch**: A cover may be sung in a different key
- **Tempo**: The speed may vary
- **Instrumentation**: Different instruments, arrangements
- **Vocal style**: Different singer, different interpretation

Yet the underlying melody and harmonic structure remain similar.

### What is CQTNet?

CQTNet is a Convolutional Neural Network (CNN) designed specifically for cover song identification.
It was originally proposed and trained on the **SHS100K** dataset (a large Western music dataset).

Key ideas:
1. **Input**: Constant-Q Transform (CQT) spectrograms of audio
2. **Architecture**: Deep CNN that progressively extracts hierarchical features
3. **Output**: A compact 300-dimensional embedding vector that captures the song's identity
4. **Training strategy**: Trained as a classification task (each original song = one class),
   but at test time we use the intermediate embedding for similarity comparison

### What Are We Doing?

In this notebook, we **train CQTNet from scratch** (random weight initialization, no pretrained Western weights)
on **Indian music** MP3 files. This means:
- We do NOT load any pretrained model
- We adapt the final classification layer to match OUR number of songs
- We train entirely on Indian music data

The notebook includes a **DEMO MODE** with synthetic data so it can run end-to-end
without requiring actual MP3 files.

### Installing Required Libraries

Below we install the Python packages needed for this project:

| Library | Purpose |
|---------|--------|
| `torch` | PyTorch deep learning framework - builds and trains the CNN |
| `torchaudio` | Audio processing utilities for PyTorch |
| `librosa` | Audio analysis library - computes CQT spectrograms |
| `numpy` | Numerical computing - array operations |
| `matplotlib` | Plotting and visualization |
| `scikit-learn` | Machine learning utilities (metrics, etc.) |
| `soundfile` | Reading/writing audio files (backend for librosa) |

**WHY these specific libraries?**
- `librosa` has a well-tested CQT implementation that matches what CQTNet was designed for
- `torch` provides automatic differentiation (backpropagation) and GPU acceleration
- `numpy` is the standard for numerical array manipulation in Python

In [ ]:
# Install required packages
# pip install is used with -q (quiet) flag to reduce output noise
# If running in Google Colab, these may already be partially installed

!pip install -q torch torchvision torchaudio
!pip install -q librosa numpy matplotlib scikit-learn soundfile

### Importing Libraries

Now we import all the modules we will use throughout this notebook.
Each import is commented to explain its role.

**WHAT**: Load all required Python modules into memory.

**WHY**: Python requires explicit imports so the interpreter knows which
functions and classes are available. Grouping them at the top is a best practice
for readability.

**HOW it connects**: Every subsequent cell depends on these imports.

In [ ]:
import os                      # Operating system interface - file/directory operations
import glob                    # Unix-style pathname pattern matching
import random                  # Random number generation for reproducibility
import warnings                # Suppress non-critical warning messages

import numpy as np             # Numerical computing (arrays, linear algebra)
import matplotlib.pyplot as plt  # Plotting library for visualizations
import librosa                 # Audio analysis (CQT computation, audio loading)
import librosa.display         # Specialized display functions for audio spectrograms

import torch                   # PyTorch core - tensors, autograd
import torch.nn as nn          # Neural network layers (Conv2d, Linear, etc.)
import torch.optim as optim    # Optimizers (Adam, SGD, etc.)
from torch.utils.data import Dataset, DataLoader  # Data loading utilities
from collections import OrderedDict  # Ordered dictionary for naming layers

from sklearn.metrics import average_precision_score  # For MAP evaluation

# Suppress librosa warnings about audio backends
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Librosa version: {librosa.__version__}")

### Setting Random Seeds for Reproducibility

**WHAT**: Fix all random number generators to produce the same sequence every run.

**WHY**: Neural network training involves many random operations:
- Random weight initialization
- Random data shuffling
- Random cropping of spectrograms

Without fixed seeds, you get different results each run, making debugging impossible
and making it hard to demonstrate that your code works.

**HOW it connects**: This must be set BEFORE creating the model or data loaders,
as those operations use random numbers during initialization.

In [ ]:
# Set seeds for ALL random number generators used in the pipeline
SEED = 42  # Any integer works; 42 is a common convention

random.seed(SEED)          # Python's built-in random module
np.random.seed(SEED)       # NumPy's random number generator
torch.manual_seed(SEED)    # PyTorch CPU random number generator

# If using GPU, also set CUDA seeds
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)       # Current GPU
    torch.cuda.manual_seed_all(SEED)   # All GPUs (if multi-GPU)

# Make PyTorch operations deterministic (slight speed cost)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {SEED}")
print("All random operations will now be reproducible.")

### Device Selection (GPU vs CPU)

**WHAT**: Detect whether a GPU (Graphics Processing Unit) is available and select it.

**WHY**: GPUs can perform matrix multiplications (the core of CNN computation)
~10-100x faster than CPUs because they have thousands of cores optimized for
parallel arithmetic. Training on CPU is possible but very slow.

**HOW it connects**: Every tensor and model must be moved to the same device.
We define `device` here and use it throughout to ensure consistency.

In [ ]:
# Check if NVIDIA GPU is available via CUDA
# CUDA = Compute Unified Device Architecture (NVIDIA's GPU computing platform)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected. Training will be slower but still works.")
    print("For Google Colab: Runtime -> Change runtime type -> GPU")

### Configuration: Demo Mode vs Real Data Mode

**WHAT**: Set a flag to control whether we use synthetic (fake) data or real MP3 files.

**WHY**: 
- **DEMO_MODE = True**: Generates synthetic data so the notebook runs end-to-end
  without needing actual audio files. Perfect for testing the pipeline.
- **DEMO_MODE = False**: Uses real Indian music MP3 files from a specified directory.

**HOW it connects**: This flag controls data generation in Section 3.
Set it to `False` when you have your real Indian music dataset ready.

In [ ]:
# ============================================================
# CONFIGURATION - Change these settings for your use case
# ============================================================

DEMO_MODE = True  # Set to False when using real MP3 data

# Path to your Indian music dataset (only used when DEMO_MODE = False)
# Expected structure:
#   DATA_DIR/
#     song_001/
#       original.mp3
#       cover_1.mp3
#     song_002/
#       original.mp3
#       cover_1.mp3
DATA_DIR = './dataset/'  # Change this to your actual data path

# Path where preprocessed CQT .npy files will be saved
CQT_DIR = './cqt_features/'

# Number of unique original songs in your dataset
# For demo: 5 synthetic songs
# For real Indian data: set to the number of unique songs (e.g., 245)
NUM_CLASSES = 5 if DEMO_MODE else 245

print(f"Mode: {'DEMO (synthetic data)' if DEMO_MODE else 'REAL DATA'}")
print(f"Number of classes (songs): {NUM_CLASSES}")
print(f"Data directory: {DATA_DIR}")
print(f"CQT output directory: {CQT_DIR}")

---
## Section 2: Understanding the Constant-Q Transform (CQT)

### What is a Spectrogram?

Raw audio is a 1D waveform (amplitude vs. time). But music perception is based on
**frequencies** (which notes are playing). A spectrogram converts audio from the
time domain to the time-frequency domain, showing which frequencies are active at each moment.

### What is CQT and How Does It Differ from STFT/Mel?

| Feature | STFT | Mel Spectrogram | CQT |
|---------|------|-----------------|-----|
| Frequency spacing | Linear (equal Hz between bins) | Mel-scaled (perceptual) | **Logarithmic (equal musical intervals)** |
| Octave representation | Uneven (higher octaves get more bins) | Somewhat even | **Perfectly even** |
| Musical relevance | Low | Medium | **High** |
| Pitch shift effect | Complex distortion | Non-uniform shift | **Simple vertical translation** |

### Why CQT for Cover Song Identification?

1. **Musical notes are logarithmically spaced**: A4=440Hz, A5=880Hz, A6=1760Hz.
   Each octave doubles in frequency. CQT respects this natural musical structure.

2. **Transposition becomes translation**: If a cover is in a different key (pitched up/down),
   in CQT this appears as a simple vertical shift. CNNs can learn to be robust to this.

3. **Equal resolution per octave**: Every octave gets the same number of frequency bins
   (default 12 bins/octave = 1 per semitone). This means equal detail for bass and treble.

The default CQT in librosa produces **84 frequency bins** (7 octaves x 12 bins/octave).

### Generating a Synthetic Test Signal

**WHAT**: Create a synthetic audio signal with known frequencies so we can
visualize and understand the CQT without needing real audio files.

**WHY**: This makes the notebook self-contained and runnable anywhere.
By using known frequencies (musical notes), we can verify the CQT is working
correctly (we should see energy at exactly those frequency bins).

**HOW it connects**: This demonstrates the CQT computation that will be applied
to all MP3 files in the dataset preparation stage.

In [ ]:
# Generate a synthetic audio signal for demonstration
# We create a signal that plays 3 notes sequentially:
#   C4 (261.6 Hz), E4 (329.6 Hz), G4 (392.0 Hz) - a C major chord arpeggio

sr = 22050  # Sample rate in Hz (samples per second)
            # 22050 is standard for music analysis (Nyquist freq = 11025 Hz)
duration = 3.0  # Total duration in seconds

# Time axis: 3 seconds at 22050 samples/sec = 66150 samples
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
print(f"Time array shape: {t.shape}")
print(f"Total samples: {len(t)} (= {sr} samples/sec x {duration} sec)")

# Generate three notes, each 1 second long
# np.sin(2 * pi * frequency * time) generates a pure sine wave
note_C4 = np.sin(2 * np.pi * 261.63 * t)  # C4 = 261.63 Hz (Middle C)
note_E4 = np.sin(2 * np.pi * 329.63 * t)  # E4 = 329.63 Hz
note_G4 = np.sin(2 * np.pi * 392.00 * t)  # G4 = 392.00 Hz

# Combine: C4 plays 0-1s, E4 plays 1-2s, G4 plays 2-3s
signal = np.zeros_like(t)
samples_per_note = int(sr * 1.0)  # 22050 samples = 1 second
signal[0:samples_per_note] = note_C4[0:samples_per_note]
signal[samples_per_note:2*samples_per_note] = note_E4[samples_per_note:2*samples_per_note]
signal[2*samples_per_note:3*samples_per_note] = note_G4[2*samples_per_note:3*samples_per_note]

# Add a tiny bit of noise to make it more realistic
signal += 0.01 * np.random.randn(len(signal))

print(f"\nSynthetic signal created:")
print(f"  Duration: {duration}s")
print(f"  Sample rate: {sr} Hz")
print(f"  Notes: C4 (261.6 Hz) -> E4 (329.6 Hz) -> G4 (392.0 Hz)")
print(f"  Signal shape: {signal.shape}")
print(f"  Signal dtype: {signal.dtype}")

### Computing the CQT

**WHAT**: Apply the Constant-Q Transform to our synthetic signal.

**WHY**: This is exactly what we will do to every MP3 file in our dataset.
Understanding the output shape and meaning is critical.

**HOW it connects**: The CQT output becomes the input to our CNN.
The CNN expects input of shape `[batch, 1, 84, time]` where 84 is the number of CQT frequency bins.

**Key parameters**:
- `sr`: Sample rate (must match the audio)
- `hop_length`: Number of samples between successive CQT columns (default 512)
- `n_bins`: Total number of frequency bins (default 84 = 7 octaves x 12 bins/octave)
- `bins_per_octave`: Frequency resolution (12 = one bin per semitone)

In [ ]:
# Compute the Constant-Q Transform
# librosa.cqt returns a COMPLEX array (magnitude + phase)
# We take np.abs() to get only the magnitude (we discard phase information)

cqt_complex = librosa.cqt(y=signal, sr=sr)  # Complex CQT
cqt_magnitude = np.abs(cqt_complex)          # Take absolute value = magnitude

print(f"CQT output shape: {cqt_magnitude.shape}")
print(f"  - Dimension 0: {cqt_magnitude.shape[0]} frequency bins (7 octaves x 12 bins/octave = 84)")
print(f"  - Dimension 1: {cqt_magnitude.shape[1]} time frames")
print(f"  - Time resolution: {duration / cqt_magnitude.shape[1] * 1000:.1f} ms per frame")
print(f"\nFrequency range:")
print(f"  - Lowest bin: {librosa.note_to_hz('C1'):.1f} Hz (C1)")
print(f"  - Highest bin: {librosa.note_to_hz('B7'):.1f} Hz (B7)")
print(f"\nData type: {cqt_magnitude.dtype}")
print(f"Value range: [{cqt_magnitude.min():.4f}, {cqt_magnitude.max():.4f}]")

### Visualizing the CQT Spectrogram

**WHAT**: Plot the CQT spectrogram to see what our CNN will 'see' as input.

**WHY**: Visual inspection helps verify:
- The CQT is computed correctly (we should see energy at our known frequencies)
- We understand what patterns the CNN needs to learn

**HOW it connects**: The CNN's convolutional filters will learn to detect patterns
in images that look exactly like this.

In [ ]:
# Visualize the CQT spectrogram
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Raw waveform
axes[0].plot(t, signal, linewidth=0.5)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Raw Audio Waveform (C4 -> E4 -> G4 arpeggio)')
axes[0].axvline(x=1.0, color='r', linestyle='--', alpha=0.5, label='Note change')
axes[0].axvline(x=2.0, color='r', linestyle='--', alpha=0.5)
axes[0].legend()

# Plot 2: CQT spectrogram
# We use amplitude_to_db to convert to decibel scale (log scale)
# This makes quiet sounds visible (human hearing is logarithmic)
cqt_db = librosa.amplitude_to_db(cqt_magnitude, ref=np.max)
img = librosa.display.specshow(cqt_db, sr=sr, x_axis='time', y_axis='cqt_note',
                                ax=axes[1])
axes[1].set_title('CQT Spectrogram (dB scale)')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Note')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

print("\nYou should see:")
print("  - Bright horizontal line at C4 during 0-1 seconds")
print("  - Bright horizontal line at E4 during 1-2 seconds")
print("  - Bright horizontal line at G4 during 2-3 seconds")
print("  - Plus harmonics (integer multiples of fundamental frequency)")

### Temporal Mean-Pooling

**WHAT**: Reduce the time dimension by averaging every 20 consecutive frames into one frame.

**WHY**:
1. **Dimensionality reduction**: A 3-minute song at ~43 frames/sec = ~7700 frames.
   After pooling with window=20: ~385 frames. Much more manageable.
2. **Smoothing**: Averages out micro-level timing variations that are not musically relevant.
3. **Memory efficiency**: Smaller arrays = less RAM and faster training.

**HOW it connects**: This is applied BEFORE saving the .npy files.
The CNN receives the POOLED CQT, not the raw CQT.

**Original CQTNet paper**: Uses `mean_size = 20` for this temporal pooling step.

In [ ]:
# Temporal mean-pooling: average every 20 consecutive time frames into 1
# This is EXACTLY how the original CQTNet (gencqt.py) preprocesses audio

mean_size = 20  # Number of frames to average together

# Original CQT shape
height, length = cqt_magnitude.shape  # height=84 (freq bins), length=time frames
print(f"Before pooling: shape = {cqt_magnitude.shape}")
print(f"  - {height} frequency bins x {length} time frames")

# Calculate new time dimension (integer division - discard remainder frames)
new_length = length // mean_size
print(f"\nPooling: averaging every {mean_size} frames")
print(f"  - {length} frames / {mean_size} = {new_length} pooled frames")
print(f"  - Discarding last {length % mean_size} frames (remainder)")

# Perform the mean-pooling
# For each new frame i, average frames [i*20 : (i+1)*20]
new_cqt = np.zeros((height, new_length), dtype=np.float64)
for i in range(new_length):
    # Take a slice of 20 frames and average along time axis
    new_cqt[:, i] = cqt_magnitude[:, i*mean_size:(i+1)*mean_size].mean(axis=1)

print(f"\nAfter pooling: shape = {new_cqt.shape}")
print(f"  - {new_cqt.shape[0]} frequency bins x {new_cqt.shape[1]} time frames")
print(f"  - Compression ratio: {length/new_length:.1f}x fewer time frames")
print(f"  - Each pooled frame represents {mean_size * 512 / sr * 1000:.0f} ms of audio")

### Faculty Q&A: CQT vs Other Representations

**Q: Why not use Mel spectrograms instead of CQT?**

A: Mel spectrograms use a perceptually-motivated frequency scale, but it is not
perfectly logarithmic. CQT has exactly logarithmic spacing, which means:
- Each octave has EXACTLY the same number of bins (12)
- A pitch shift of N semitones = a shift of exactly N bins vertically
- This geometric regularity makes it easier for CNNs to learn pitch-invariant features

Mel spectrograms are better for speech tasks; CQT is better for music tasks.

**Q: Why take the magnitude and discard phase?**

A: Phase contains timing information that is highly variable between recordings.
Two covers of the same song will have completely different phase patterns.
Magnitude tells us WHAT frequencies are present (the "what"), while phase tells us
the precise alignment (the "when"). For cover song identification, we care about
"what notes are played" not "exactly when each cycle starts."

**Q: Why 84 bins? Could we use more?**

A: 84 = 7 octaves x 12 bins per octave. 12 bins per octave means one bin per
semitone (the smallest interval in Western/Indian music). 7 octaves covers
C1 to B7, which encompasses virtually all musical instruments. You could use
more bins per octave (e.g., 24 for quarter-tone resolution), but 12 has been
found sufficient for cover song identification.

---
## Section 3: Dataset Preparation

### How PyTorch Datasets Work

PyTorch provides a standardized way to load data through two key classes:

1. **`Dataset`** (what we define): Tells PyTorch how to access individual samples
   - `__len__()`: Returns total number of samples
   - `__getitem__(idx)`: Returns one sample (feature + label) given an index

2. **`DataLoader`** (built-in): Handles batching, shuffling, and parallel loading
   - Calls our `__getitem__` multiple times to form a batch
   - Shuffles data order each epoch (important for training!)
   - Can use multiple CPU workers for parallel data loading

### Expected Folder Structure for Indian Music Data

```
dataset/
  song_001/           # Folder for original song #1
    original.mp3      # The original recording
    cover_1.mp3       # Cover version 1
    cover_2.mp3       # Cover version 2
  song_002/           # Folder for original song #2
    original.mp3
    cover_1.mp3
  ...
```

Each folder represents one *class* (one original song). All files within a folder
are different versions/covers of that same song. The model learns that all files
in the same folder should produce similar embeddings.

### CQT Feature Extraction from MP3 Files

**WHAT**: Walk through the dataset directory, compute CQT for each MP3 file,
apply temporal mean-pooling, and save as .npy files.

**WHY**: Computing CQT on-the-fly during training would be very slow.
Pre-computing and saving as .npy files means we only do it once,
and training can load data almost instantly.

**HOW it connects**: This creates the .npy files that our Dataset class will load.
Each .npy file is named `{class_id}_{version_id}.npy` following the original CQTNet convention.

In [ ]:
def compute_cqt_features(mp3_path, sr=22050, mean_size=20):
    """
    Compute CQT features for a single audio file.
    
    This follows EXACTLY the original CQTNet preprocessing (gencqt.py):
    1. Load audio at 22050 Hz
    2. Compute magnitude CQT (84 frequency bins)
    3. Apply temporal mean-pooling with window of 20 frames
    
    Parameters:
    -----------
    mp3_path : str
        Path to the audio file (MP3, WAV, FLAC, etc.)
    sr : int
        Target sample rate. Default 22050 Hz (librosa default).
    mean_size : int
        Number of CQT frames to average. Default 20.
    
    Returns:
    --------
    new_cqt : np.ndarray
        Shape: (84, T) where T = num_frames // mean_size
        The temporally-pooled CQT magnitude spectrogram.
    """
    # Step 1: Load audio file, resample to target sr
    # librosa.load returns: (audio_array, sample_rate)
    # It automatically: converts stereo to mono, resamples to `sr`
    data, sr = librosa.load(mp3_path, sr=sr)
    # data shape: (num_samples,) - 1D array
    
    # Step 2: Compute CQT magnitude
    # np.abs converts complex CQT to magnitude (discards phase)
    cqt = np.abs(librosa.cqt(y=data, sr=sr))
    # cqt shape: (84, num_frames) where num_frames depends on audio length
    
    # Step 3: Temporal mean-pooling
    # Average every `mean_size` consecutive frames into one frame
    height, length = cqt.shape  # height=84, length=num_frames
    new_length = length // mean_size  # Integer division
    
    new_cqt = np.zeros((height, new_length), dtype=np.float64)
    for i in range(new_length):
        new_cqt[:, i] = cqt[:, i*mean_size:(i+1)*mean_size].mean(axis=1)
    # new_cqt shape: (84, new_length)
    
    return new_cqt


print("Function defined: compute_cqt_features()")
print("  Input: path to audio file")
print("  Output: numpy array of shape (84, T)")

### Processing the Dataset (Real Mode) or Generating Synthetic Data (Demo Mode)

**WHAT**: Either process real MP3 files into .npy features, or generate synthetic
data that mimics the structure of real CQT features.

**WHY for Demo Mode**: We need synthetic data that:
- Has the same shape as real CQT data: (84, T)
- Has class structure: different "songs" should have different patterns
- Has within-class similarity: "covers" of the same song should be similar
- Allows the notebook to run end-to-end without real audio files

**HOW it connects**: After this cell, we have .npy files on disk
ready to be loaded by our PyTorch Dataset class.

In [ ]:
os.makedirs(CQT_DIR, exist_ok=True)  # Create output directory if it doesn't exist

if DEMO_MODE:
    # ============================================================
    # DEMO MODE: Generate synthetic CQT-like data
    # ============================================================
    # We create 5 synthetic "songs" with 3 "covers" each = 15 total files
    # Each song has a unique base pattern that covers share (with noise added)
    
    NUM_SONGS = 5        # Number of unique songs (classes)
    COVERS_PER_SONG = 3  # Number of versions per song
    TIME_FRAMES = 300    # Temporal dimension (similar to ~30 seconds of audio)
    FREQ_BINS = 84       # CQT frequency bins (7 octaves x 12 bins/octave)
    
    print(f"Generating synthetic data:")
    print(f"  {NUM_SONGS} songs x {COVERS_PER_SONG} covers = {NUM_SONGS * COVERS_PER_SONG} total files")
    print(f"  Each file: shape ({FREQ_BINS}, {TIME_FRAMES})")
    print()
    
    file_list = []  # Will store (filepath, class_label) pairs
    
    for song_id in range(NUM_SONGS):
        # Create a unique base pattern for this "song"
        # This simulates the harmonic structure unique to each song
        np.random.seed(song_id * 100)  # Deterministic per song
        base_pattern = np.random.rand(FREQ_BINS, TIME_FRAMES) * 0.5
        
        # Add some "melodic structure" - activate certain frequency bands
        # This simulates the fact that songs have specific note patterns
        active_bins = np.random.choice(FREQ_BINS, size=10, replace=False)
        for b in active_bins:
            base_pattern[b, :] += np.random.rand(TIME_FRAMES) * 2.0
        
        for cover_id in range(COVERS_PER_SONG):
            # Each cover is the base pattern + random noise
            # This simulates how covers share harmonic structure but differ in details
            noise = np.random.rand(FREQ_BINS, TIME_FRAMES) * 0.3
            cover_cqt = base_pattern + noise
            
            # Ensure non-negative (real CQT magnitudes are always >= 0)
            cover_cqt = np.maximum(cover_cqt, 0)
            
            # Save with naming convention: {song_id}_{cover_id}.npy
            filename = f"{song_id}_{cover_id}.npy"
            filepath = os.path.join(CQT_DIR, filename)
            np.save(filepath, cover_cqt)
            file_list.append((filepath, song_id))
            
            print(f"  Created: {filename} (song {song_id}, cover {cover_id}) "
                  f"shape={cover_cqt.shape}")
    
    # Reset random seed
    np.random.seed(SEED)
    
    print(f"\nTotal synthetic files created: {len(file_list)}")

else:
    # ============================================================
    # REAL DATA MODE: Process actual MP3 files
    # ============================================================
    file_list = []  # Will store (filepath, class_label) pairs
    
    # Get sorted list of song directories
    song_dirs = sorted([d for d in os.listdir(DATA_DIR) 
                        if os.path.isdir(os.path.join(DATA_DIR, d))])
    print(f"Found {len(song_dirs)} song directories")
    
    for song_id, song_dir in enumerate(song_dirs):
        song_path = os.path.join(DATA_DIR, song_dir)
        
        # Find all MP3 files in this song's directory
        mp3_files = glob.glob(os.path.join(song_path, '*.mp3'))
        
        for cover_id, mp3_path in enumerate(sorted(mp3_files)):
            # Compute CQT features
            try:
                cqt_features = compute_cqt_features(mp3_path)
                
                # Save as .npy file
                filename = f"{song_id}_{cover_id}.npy"
                filepath = os.path.join(CQT_DIR, filename)
                np.save(filepath, cqt_features)
                file_list.append((filepath, song_id))
                
                print(f"  Processed: {os.path.basename(mp3_path)} -> {filename} "
                      f"shape={cqt_features.shape}")
            except Exception as e:
                print(f"  ERROR processing {mp3_path}: {e}")
    
    print(f"\nTotal files processed: {len(file_list)}")
    NUM_CLASSES = len(song_dirs)
    print(f"Number of classes (unique songs): {NUM_CLASSES}")

### PyTorch Dataset Class

**WHAT**: Define a custom Dataset class that loads preprocessed CQT .npy files
and prepares them for the CNN.

**WHY**: PyTorch's DataLoader needs a Dataset object to know:
- How many samples exist (`__len__`)
- How to get one sample (`__getitem__`)

**HOW it connects**: The DataLoader will call `__getitem__` in a loop to build
batches of data for training.

**Transforms applied to each sample**:
1. **Transpose**: CQT is stored as (84, T) but we need (T, 84) for random cropping
2. **Normalization**: Subtract mean, divide by std. This helps gradient-based training.
3. **Random crop/pad**: Make all samples the same temporal length (out_length).
   - If sample is longer: randomly select a contiguous window of out_length frames.
   - If sample is shorter: pad with zeros.
4. **Reshape**: Final shape is (1, 84, out_length) - the 1 is the "channel" dimension
   (like grayscale images have 1 channel vs RGB's 3 channels)

In [ ]:
class CQTDataset(Dataset):
    """
    PyTorch Dataset for loading preprocessed CQT features.
    
    Each .npy file contains a CQT spectrogram of shape (84, T)
    where T varies per file (different songs have different lengths).
    
    The Dataset handles:
    1. Loading from disk
    2. Transposing to (T, 84)
    3. Normalizing (zero mean, unit variance)
    4. Random cropping or padding to fixed length
    5. Reshaping to (1, 84, out_length) for CNN input
    """
    
    def __init__(self, file_list, out_length=200, is_training=True):
        """
        Parameters:
        -----------
        file_list : list of tuples
            Each element is (filepath, class_label)
            filepath: path to .npy file
            class_label: integer class ID (which song)
        out_length : int
            Fixed temporal length for output. Default 200 frames.
            Original CQTNet uses 200, 300, or 400 during multi-size training.
        is_training : bool
            If True: random crop. If False: center crop (for evaluation).
        """
        self.file_list = file_list      # List of (path, label) tuples
        self.out_length = out_length    # Target temporal dimension
        self.is_training = is_training  # Controls random vs center crop
    
    def __len__(self):
        """Return total number of samples in dataset."""
        # DataLoader calls this to know how many batches per epoch
        return len(self.file_list)
    
    def __getitem__(self, idx):
        """
        Load and preprocess one sample.
        
        Parameters:
        -----------
        idx : int
            Index into self.file_list
        
        Returns:
        --------
        data_tensor : torch.FloatTensor
            Shape: (1, 84, out_length)
            The preprocessed CQT ready for the CNN
        label : int
            The class label (song ID)
        """
        filepath, label = self.file_list[idx]
        
        # Step 1: Load the .npy file
        # Shape: (84, T) where T is the original time length
        cqt = np.load(filepath)
        
        # Step 2: Transpose from (84, T) to (T, 84)
        # WHY: Makes random cropping along time axis (axis 0) easier
        cqt = cqt.T  # Now shape: (T, 84)
        
        # Step 3: Normalize to zero mean and unit variance
        # WHY: Neural networks train better when inputs are centered around 0
        # with standard deviation of 1. Without this, different songs with
        # different volume levels would confuse the network.
        mean = cqt.mean()
        std = cqt.std()
        if std > 0:  # Avoid division by zero
            cqt = (cqt - mean) / std
        # Shape still: (T, 84)
        
        # Step 4: Random crop or pad to fixed length
        T = cqt.shape[0]  # Current temporal length
        
        if T >= self.out_length:
            # Song is long enough: crop a random window
            if self.is_training:
                # Random start position (data augmentation)
                start = random.randint(0, T - self.out_length)
            else:
                # Center crop (deterministic for evaluation)
                start = (T - self.out_length) // 2
            cqt = cqt[start:start + self.out_length, :]  # Shape: (out_length, 84)
        else:
            # Song is too short: pad with zeros at the end
            # WHY zeros: maintains the normalized distribution
            padding = np.zeros((self.out_length - T, 84))
            cqt = np.concatenate([cqt, padding], axis=0)  # Shape: (out_length, 84)
        
        # Step 5: Transpose back to (84, out_length) and add channel dimension
        cqt = cqt.T  # Shape: (84, out_length)
        cqt = cqt[np.newaxis, :, :]  # Shape: (1, 84, out_length)
        # The 1 = number of channels (like grayscale images)
        
        # Step 6: Convert to PyTorch tensor
        data_tensor = torch.FloatTensor(cqt)  # Convert numpy -> pytorch tensor
        
        return data_tensor, label


print("CQTDataset class defined.")
print("  Input: list of (filepath, label) tuples")
print("  Output per sample: tensor of shape (1, 84, out_length), label")

### Creating Train/Test Split and DataLoaders

**WHAT**: Split our data into training and testing sets, then wrap them in DataLoaders.

**WHY**:
- **Train set**: Used to update model weights (the model learns from this)
- **Test set**: Used to evaluate performance (the model never learns from this)
- We need both to check if the model generalizes (works on unseen data)

**HOW it connects**: DataLoaders are passed directly to the training loop.

In [ ]:
# Split data into train and test sets
# We use a simple split: for each song, use 2 covers for training, 1 for testing
# (In demo mode: 3 covers per song -> 2 train + 1 test)

train_files = []
test_files = []

# Group files by class label
from collections import defaultdict
class_files = defaultdict(list)
for filepath, label in file_list:
    class_files[label].append((filepath, label))

# For each class, put last file in test, rest in train
for label in sorted(class_files.keys()):
    files = class_files[label]
    if len(files) > 1:
        train_files.extend(files[:-1])  # All except last -> train
        test_files.extend(files[-1:])   # Last one -> test
    else:
        train_files.extend(files)  # If only 1 file, put in train

print(f"Dataset split:")
print(f"  Training samples: {len(train_files)}")
print(f"  Testing samples:  {len(test_files)}")
print(f"  Total:            {len(train_files) + len(test_files)}")

# Create Dataset objects
OUT_LENGTH = 200  # Fixed temporal length for this training run
                  # Original CQTNet uses 200, 300, 400 in multi-size training

train_dataset = CQTDataset(train_files, out_length=OUT_LENGTH, is_training=True)
test_dataset = CQTDataset(test_files, out_length=OUT_LENGTH, is_training=False)

# Create DataLoaders
# batch_size: how many samples per gradient update
# shuffle: randomize order each epoch (True for train, False for test)
# num_workers: parallel data loading processes (0 = main process only)
BATCH_SIZE = 4  # Small batch for demo; use 32-64 for real training

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # Shuffle training data each epoch
    num_workers=0,       # Set to 2-4 for faster loading on real data
    drop_last=True       # Drop incomplete last batch (helps BatchNorm stability)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # No need to shuffle test data
    num_workers=0
)

print(f"\nDataLoader created:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training batches per epoch: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

### Verifying the Dataset Output

**WHAT**: Load one batch and inspect its shape to confirm everything works.

**WHY**: Always verify data pipeline before training! Shape mismatches are
the most common bug in deep learning.

**Expected shape**: `(batch_size, 1, 84, out_length)` = `(4, 1, 84, 200)`

In [ ]:
# Get one batch to verify shapes
sample_batch, sample_labels = next(iter(train_loader))

print(f"Sample batch shape: {sample_batch.shape}")
print(f"  - Dimension 0: {sample_batch.shape[0]} (batch size)")
print(f"  - Dimension 1: {sample_batch.shape[1]} (channels = 1, like grayscale)")
print(f"  - Dimension 2: {sample_batch.shape[2]} (frequency bins = 84)")
print(f"  - Dimension 3: {sample_batch.shape[3]} (time frames = {OUT_LENGTH})")
print(f"\nSample labels: {sample_labels}")
print(f"Labels dtype: {sample_labels.dtype}")
print(f"\nBatch statistics:")
print(f"  Min value:  {sample_batch.min():.3f}")
print(f"  Max value:  {sample_batch.max():.3f}")
print(f"  Mean value: {sample_batch.mean():.3f}")
print(f"  Std value:  {sample_batch.std():.3f}")

---
## Section 4: CQTNet Architecture (Detailed)

### CNN Basics for Audio

Before building CQTNet, let us review the core building blocks:

#### Convolution (Conv2d)
- A small filter (kernel) slides across the input, computing dot products
- **What it learns**: Local patterns (e.g., specific frequency-time shapes = musical patterns)
- **Parameters**: kernel_size, number of filters (out_channels), dilation, padding

#### Batch Normalization (BatchNorm2d)
- Normalizes activations across the batch to have mean=0, std=1
- Then applies learnable scale (gamma) and shift (beta)
- **Why needed**: Prevents internal covariate shift, allows higher learning rates,
  acts as mild regularization

#### ReLU Activation
- `f(x) = max(0, x)` - simply zeroes out negative values
- **Why needed**: Introduces non-linearity. Without it, stacking layers is useless
  (a stack of linear layers = one linear layer). Sparse activations also help efficiency.

#### MaxPool2d
- Takes the maximum value in a local window
- **What it does**: Downsamples (reduces dimensions), adds translation invariance
- In CQTNet: MaxPool2d((1,2)) pools ONLY in time dimension, not frequency

#### Dilation
- Inserts gaps between kernel elements
- dilation=(1,2) means: in the time dimension, skip every other position
- **Why used**: Increases receptive field (how much context each neuron sees)
  without increasing parameter count. A 3x3 kernel with dilation=2 covers
  the same area as a 3x5 kernel but with fewer parameters.

#### AdaptiveMaxPool2d((1,1))
- Regardless of input spatial size, outputs exactly 1x1
- **Why needed**: Allows the network to accept variable-length inputs.
  Whether input is 200, 300, or 400 frames, output is always 1x1x512.

### Building the CQTNet Model

**WHAT**: Define the complete CQTNet architecture as a PyTorch nn.Module.

**WHY**: This is the exact same architecture used in the original CQTNet paper,
with ONE modification: `fc1` outputs `num_classes` instead of 10000.
The original used 10000 because SHS100K has ~10000 songs.
We use `num_classes` as a variable because our Indian dataset has fewer songs.

**HOW it connects**: This model will be instantiated, moved to GPU,
and trained with our DataLoader.

**Architecture overview**:
```
Input: (batch, 1, 84, time)
  -> 10 Conv blocks (conv + batchnorm + relu, with maxpool every 2 blocks)
  -> AdaptiveMaxPool to (batch, 512, 1, 1)
  -> Flatten to (batch, 512)
  -> FC to (batch, 300) <- THIS is the embedding we use at test time
  -> FC to (batch, num_classes) <- THIS is for classification loss during training
```

In [ ]:
class CQTNet(nn.Module):
    """
    CQTNet: A CNN for cover song identification using CQT spectrograms.
    
    Architecture: 10 convolutional layers + 2 fully-connected layers
    Input shape:  (batch_size, 1, 84, time_frames)
    Output:       (classification_logits, 300-dim_embedding)
    
    The key insight: we train with classification loss but USE the 300-dim
    embedding for cover song retrieval at test time.
    """
    
    def __init__(self, num_classes=5):
        """
        Parameters:
        -----------
        num_classes : int
            Number of unique songs (classes) in the dataset.
            Original CQTNet uses 10000 (for SHS100K dataset).
            Set this to YOUR number of songs.
        """
        super(CQTNet, self).__init__()
        
        # ============================================================
        # FEATURE EXTRACTION LAYERS (Convolutional backbone)
        # ============================================================
        # Using OrderedDict so each layer has a descriptive name
        # This helps with debugging and model inspection
        
        self.features = nn.Sequential(OrderedDict([
            # ---- Block 1: Initial feature extraction ----
            # Input shape: (batch, 1, 84, time)
            
            # Conv0: First convolution layer
            # kernel_size=(12, 3): 
            #   - 12 in frequency = spans ~1.5 octaves (12 bins = 1 octave in CQT)
            #   - 3 in time = looks at 3 consecutive time frames
            # WHY 12 in frequency? To capture patterns that span more than one octave
            #   (e.g., harmonics, octave relationships between notes)
            # padding=(6, 0): pad 6 on each side of frequency to maintain freq dimension
            # bias=False: no bias because BatchNorm has its own bias (beta parameter)
            ('conv0', nn.Conv2d(1, 32, kernel_size=(12, 3), dilation=(1, 1),
                                padding=(6, 0), bias=False)),
            # Output shape: (batch, 32, 84+12-12+1=85, time-2)
            # Actually with padding=(6,0): (batch, 32, 84, time-2)
            
            ('norm0', nn.BatchNorm2d(32)),   # Normalize 32 feature maps
            ('relu0', nn.ReLU(inplace=True)),  # inplace=True saves memory
            
            # Conv1: Second convolution
            # kernel_size=(13, 3) with dilation=(1, 2):
            #   - 13 in frequency: slightly larger than one octave
            #   - 3 in time with dilation=2: effective receptive field = 5 in time
            # WHY dilation=(1,2)? Increases temporal context without more parameters
            #   Effective kernel: sees frames [t, t+2, t+4] instead of [t, t+1, t+2]
            ('conv1', nn.Conv2d(32, 64, kernel_size=(13, 3), dilation=(1, 2),
                                bias=False)),
            ('norm1', nn.BatchNorm2d(64)),
            ('relu1', nn.ReLU(inplace=True)),
            
            # MaxPool1: Downsample time dimension by factor 2
            # kernel=(1,2), stride=(1,2): only pools in TIME, not frequency
            # WHY only time? We want to preserve full frequency resolution
            # padding=(0,1): handles odd time dimensions
            ('pool1', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),
            
            # ---- Block 2 ----
            # Conv2: kernel=(13,3), no dilation
            ('conv2', nn.Conv2d(64, 64, kernel_size=(13, 3), dilation=(1, 1),
                                bias=False)),
            ('norm2', nn.BatchNorm2d(64)),
            ('relu2', nn.ReLU(inplace=True)),
            
            # Conv3: kernel=(3,3) with dilation=(1,2)
            # Smaller frequency kernel now (3) - fine-grained frequency patterns
            ('conv3', nn.Conv2d(64, 64, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm3', nn.BatchNorm2d(64)),
            ('relu3', nn.ReLU(inplace=True)),
            ('pool3', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),
            
            # ---- Block 3: Increase channels to 128 ----
            # More channels = more different patterns can be detected
            ('conv4', nn.Conv2d(64, 128, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm4', nn.BatchNorm2d(128)),
            ('relu4', nn.ReLU(inplace=True)),
            
            ('conv5', nn.Conv2d(128, 128, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm5', nn.BatchNorm2d(128)),
            ('relu5', nn.ReLU(inplace=True)),
            ('pool5', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),
            
            # ---- Block 4: Increase channels to 256 ----
            ('conv6', nn.Conv2d(128, 256, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm6', nn.BatchNorm2d(256)),
            ('relu6', nn.ReLU(inplace=True)),
            
            ('conv7', nn.Conv2d(256, 256, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm7', nn.BatchNorm2d(256)),
            ('relu7', nn.ReLU(inplace=True)),
            ('pool7', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),
            
            # ---- Block 5: Increase channels to 512 ----
            # Deepest features - most abstract representations
            ('conv8', nn.Conv2d(256, 512, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm8', nn.BatchNorm2d(512)),
            ('relu8', nn.ReLU(inplace=True)),
            
            # Final conv layer
            ('conv9', nn.Conv2d(512, 512, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm9', nn.BatchNorm2d(512)),
            ('relu9', nn.ReLU(inplace=True)),
        ]))
        
        # ============================================================
        # GLOBAL POOLING
        # ============================================================
        # AdaptiveMaxPool2d((1, 1)): regardless of input spatial dimensions,
        # output is ALWAYS (batch, 512, 1, 1)
        # This is what allows variable-length input!
        # A 200-frame input and a 400-frame input both produce (batch, 512, 1, 1)
        self.pool = nn.AdaptiveMaxPool2d((1, 1))
        
        # ============================================================
        # FULLY CONNECTED LAYERS (Embedding + Classification head)
        # ============================================================
        
        # fc0: Embedding layer
        # Compresses 512 features into 300-dimensional embedding
        # THIS is the layer whose output we use for cover song retrieval
        # WHY 300? Trade-off between expressiveness and compactness
        self.fc0 = nn.Linear(512, 300)
        
        # fc1: Classification head
        # Maps 300-dim embedding to class probabilities
        # num_classes = number of unique songs
        # ONLY used during training (provides the learning signal)
        # At test time, we extract the 300-dim embedding from fc0 and ignore fc1
        self.fc1 = nn.Linear(300, num_classes)
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Parameters:
        -----------
        x : torch.Tensor
            Shape: (batch_size, 1, 84, time_frames)
            The input CQT spectrogram batch
        
        Returns:
        --------
        logits : torch.Tensor
            Shape: (batch_size, num_classes)
            Raw classification scores (before softmax)
            Used with CrossEntropyLoss during training
        embedding : torch.Tensor
            Shape: (batch_size, 300)
            The learned embedding vector
            Used for cover song retrieval at test time
        """
        N = x.size(0)  # Batch size
        # x shape: (N, 1, 84, time)
        
        # Pass through all convolutional layers
        x = self.features(x)
        # x shape: (N, 512, H', W') where H' and W' depend on input size
        
        # Global adaptive max pooling
        x = self.pool(x)
        # x shape: (N, 512, 1, 1) - always, regardless of input time length
        
        # Flatten: remove spatial dimensions
        x = x.view(N, -1)
        # x shape: (N, 512)
        
        # Embedding layer
        embedding = self.fc0(x)
        # embedding shape: (N, 300) - THIS is what we use for retrieval
        
        # Classification layer
        logits = self.fc1(embedding)
        # logits shape: (N, num_classes) - for training loss
        
        return logits, embedding


print("CQTNet class defined successfully.")

### Model Summary and Parameter Count

**WHAT**: Instantiate the model and inspect its structure, counting total parameters.

**WHY**: Understanding parameter count helps you:
- Estimate memory requirements
- Understand model complexity
- Identify potential overfitting risk (too many params for small data)

In [ ]:
# Create the model
model = CQTNet(num_classes=NUM_CLASSES)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"CQTNet Model Summary")
print(f"=" * 60)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (approx): {total_params * 4 / 1024 / 1024:.1f} MB (float32)")
print(f"=" * 60)
print()

# Print layer-by-layer parameter count
print(f"{'Layer':<30} {'Shape':<25} {'Parameters':>12}")
print("-" * 70)
for name, param in model.named_parameters():
    print(f"{name:<30} {str(list(param.shape)):<25} {param.numel():>12,}")

### Shape Tracking: Following a Tensor Through the Network

**WHAT**: Pass a dummy input through the model and print the shape at every stage.

**WHY**: This is CRITICAL for understanding how data flows through the network.
Shape mismatches are the #1 debugging issue in deep learning.

**HOW it connects**: Confirms our model is correctly configured for our input shape.

In [ ]:
# Create a dummy input tensor matching our data shape
# Shape: (batch=1, channels=1, freq_bins=84, time_frames=200)
dummy_input = torch.randn(1, 1, 84, 200)
print(f"Input shape: {dummy_input.shape}")
print(f"  (batch=1, channels=1, freq=84, time=200)")
print()

# Manually pass through each layer to track shapes
x = dummy_input
print(f"{'Layer':<25} {'Output Shape':<25} {'Notes'}")
print("=" * 80)

for name, layer in model.features.named_children():
    x = layer(x)
    if 'conv' in name or 'pool' in name:
        notes = ""
        if 'conv' in name:
            # Get kernel size for context
            notes = f"kernel={layer.kernel_size}, dilation={layer.dilation}"
        elif 'pool' in name:
            notes = f"stride={layer.stride}"
        print(f"{name:<25} {str(list(x.shape)):<25} {notes}")

print()
# Adaptive pool
x = model.pool(x)
print(f"{'AdaptiveMaxPool2d':<25} {str(list(x.shape)):<25} Forces output to 1x1")

# Flatten
x = x.view(1, -1)
print(f"{'Flatten':<25} {str(list(x.shape)):<25} Remove spatial dims")

# FC layers
embedding = model.fc0(x)
print(f"{'fc0 (embedding)':<25} {str(list(embedding.shape)):<25} THE EMBEDDING")

logits = model.fc1(embedding)
print(f"{'fc1 (classifier)':<25} {str(list(logits.shape)):<25} For training loss")

print()
print(f"Final embedding dimension: 300")
print(f"Final output classes: {NUM_CLASSES}")

### Faculty Q&A: Model Architecture

**Q: Why no bias when using BatchNorm? (bias=False in all Conv layers)**

A: BatchNorm applies the transform: `output = gamma * (normalized_input) + beta`.
The `beta` parameter serves the same purpose as a bias term. If we also had bias
in the convolution, we would have TWO bias terms, which is redundant:
`Conv_bias + BN_beta` -- the network would just learn to put all the bias in one
and zero the other. Removing Conv bias saves parameters without losing expressiveness.

**Q: Why does AdaptiveMaxPool allow variable-length inputs?**

A: Regular MaxPool has a fixed kernel and stride, so output size depends on input size.
AdaptiveMaxPool works backwards: you specify the DESIRED output size (1x1),
and it automatically computes the required kernel/stride to achieve that.
So whether the input is 5x10 or 3x50, output is always 1x1.
This means our CNN can accept ANY temporal length!

**Q: What does the 300-dim embedding actually represent?**

A: It is a learned compact representation of the song's identity -- its harmonic
structure, melodic patterns, and chord progressions, all encoded into 300 numbers.
Two covers of the same song should have SIMILAR 300-dim vectors (close in space),
while different songs should have DIFFERENT vectors (far apart in space).
The classification training forces the network to learn this separation.

**Q: Why not use triplet loss from the start?**

A: Classification loss (cross-entropy) is simpler and more stable to train:
- No mining needed (no choosing of positive/negative pairs)
- Clear learning signal (is this song X? yes/no)
- Works well as a baseline
Triplet loss could improve results but adds complexity (hard negative mining,
margin selection). The original CQTNet achieved state-of-the-art with just
cross-entropy, proving it is sufficient.

---
## Section 5: Training Pipeline

### Understanding the Training Components

#### Cross-Entropy Loss
Cross-entropy measures how different our predicted probability distribution is
from the true distribution (one-hot encoded labels).

Formula: `Loss = -log(P(correct_class))`

- If model predicts 90% probability for the correct class: loss = -log(0.9) = 0.10 (low)
- If model predicts 10% probability for the correct class: loss = -log(0.1) = 2.30 (high)

The model learns to minimize this by assigning high probability to the correct song.

#### Why Classification for Embeddings?
The "trick" of CQTNet:
1. **During training**: We train as a classifier (predict which song)
2. **During testing**: We IGNORE the classifier output and instead take the
   300-dim embedding from fc0

**Why this works**: To correctly classify songs, fc0 must learn to produce
SIMILAR embeddings for covers of the same song and DIFFERENT embeddings for
different songs. This is exactly what we need for retrieval!

#### Adam Optimizer
Adam = Adaptive Moment Estimation. It maintains:
- A running average of gradients (momentum - helps with noisy gradients)
- A running average of squared gradients (adaptive learning rate per parameter)

Result: Fast convergence, works well with default settings for most tasks.

#### Learning Rate Scheduling
ReduceLROnPlateau: If the loss stops improving for `patience` epochs,
reduce the learning rate by a `factor`. This helps escape flat regions
of the loss landscape.

### Define Training Hyperparameters

**WHAT**: Set all training configuration values.

**WHY**: Hyperparameters control the training process. They are NOT learned
by the model -- we choose them based on experience and experimentation.

**HOW it connects**: These values are used in the training loop below.

In [ ]:
# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================

# Number of times to iterate over the entire dataset
# For demo: 10 epochs (fast). For real training: 50-200 epochs.
NUM_EPOCHS = 10 if DEMO_MODE else 100

# Learning rate: how big of a step to take in gradient direction
# Too high -> unstable training (loss explodes)
# Too low -> very slow convergence
# 1e-3 (0.001) is a good default for Adam
LEARNING_RATE = 1e-3

# Weight decay: L2 regularization strength
# Penalizes large weights to prevent overfitting
# Adds term: weight_decay * ||weights||^2 to the loss
WEIGHT_DECAY = 1e-4

# Learning rate scheduler settings
# Reduce LR when loss plateaus (stops improving)
LR_PATIENCE = 5    # Wait this many epochs before reducing
LR_FACTOR = 0.5    # Multiply LR by this factor (halve it)

print(f"Training Configuration:")
print(f"  Epochs:        {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay:  {WEIGHT_DECAY}")
print(f"  LR patience:   {LR_PATIENCE} epochs")
print(f"  LR factor:     {LR_FACTOR}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Device:        {device}")

### Initialize Model, Loss, Optimizer, and Scheduler

**WHAT**: Create all training components and move model to the compute device.

**WHY**: These are the four pillars of any training loop:
1. **Model**: The neural network with randomly initialized weights
2. **Loss function**: Measures how wrong the predictions are
3. **Optimizer**: Updates weights to reduce the loss
4. **Scheduler**: Adjusts learning rate during training

**HOW it connects**: All four are used together in the training loop.

In [ ]:
# Step 1: Create model and move to device (GPU or CPU)
# All weights are randomly initialized (training from scratch!)
model = CQTNet(num_classes=NUM_CLASSES).to(device)
print(f"Model created with {NUM_CLASSES} output classes")
print(f"Model moved to: {device}")
print(f"All weights randomly initialized (from scratch - no pretrained weights!)")

# Step 2: Define loss function
# CrossEntropyLoss combines LogSoftmax + NLLLoss in one step
# It expects: predictions of shape (batch, num_classes) and labels of shape (batch,)
criterion = nn.CrossEntropyLoss()
print(f"\nLoss function: CrossEntropyLoss")

# Step 3: Define optimizer
# Adam optimizer with weight decay (L2 regularization)
# model.parameters() gives Adam access to all learnable weights
optimizer = optim.Adam(
    model.parameters(),      # What to optimize (all model weights)
    lr=LEARNING_RATE,        # Initial learning rate
    weight_decay=WEIGHT_DECAY  # L2 regularization strength
)
print(f"Optimizer: Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")

# Step 4: Define learning rate scheduler
# Reduces LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',          # Reduce LR when metric DECREASES (min loss = better)
    factor=LR_FACTOR,    # Multiply LR by this factor
    patience=LR_PATIENCE,  # Wait this many epochs before reducing
    verbose=True         # Print message when LR is reduced
)
print(f"Scheduler: ReduceLROnPlateau (patience={LR_PATIENCE}, factor={LR_FACTOR})")

### The Training Loop

**WHAT**: The main training loop that iterates over epochs and batches.

**WHY**: This is where the model actually LEARNS. Each iteration:
1. Forward pass: compute predictions
2. Compute loss: how wrong are we?
3. Backward pass: compute gradients (which direction to adjust weights)
4. Optimizer step: actually adjust the weights

**HOW it connects**: After training, the model's weights encode knowledge about
our Indian music dataset, and we can extract embeddings for cover song identification.

**Expected behavior for demo**: Loss should decrease over 10 epochs,
showing the model is learning the synthetic patterns.

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

# Store loss history for plotting later
train_losses = []  # Average loss per epoch
lr_history = []    # Learning rate per epoch

print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"{'Epoch':<8} {'Loss':<12} {'Accuracy':<12} {'LR':<12}")
print("-" * 44)

for epoch in range(NUM_EPOCHS):
    # ---- Set model to training mode ----
    # This enables dropout and batch norm training behavior
    # BatchNorm: uses batch statistics for normalization
    model.train()
    
    # Accumulators for this epoch
    epoch_loss = 0.0      # Sum of all batch losses
    correct = 0           # Number of correct predictions
    total = 0             # Total number of samples
    
    # ---- Iterate over batches ----
    for batch_idx, (data, labels) in enumerate(train_loader):
        # data shape: (batch_size, 1, 84, out_length) - the CQT spectrograms
        # labels shape: (batch_size,) - integer class labels
        
        # Move data to device (GPU/CPU)
        # All tensors in a computation must be on the same device
        data = data.to(device)      # Shape: (B, 1, 84, 200)
        labels = labels.to(device)  # Shape: (B,)
        
        # ---- Step 1: Zero the gradients ----
        # WHY: PyTorch ACCUMULATES gradients by default.
        # Without zeroing, gradients from previous batch add to current batch.
        # We want fresh gradients for each batch.
        optimizer.zero_grad()
        
        # ---- Step 2: Forward pass ----
        # Pass input through the network to get predictions
        logits, embedding = model(data)
        # logits shape: (B, num_classes) - raw scores for each class
        # embedding shape: (B, 300) - not used during training, only for retrieval
        
        # ---- Step 3: Compute loss ----
        # CrossEntropyLoss internally applies softmax to logits,
        # then computes negative log-likelihood
        loss = criterion(logits, labels)
        # loss is a scalar tensor (single number)
        
        # ---- Step 4: Backward pass (backpropagation) ----
        # Compute gradient of loss with respect to EVERY learnable parameter
        # This uses the chain rule of calculus, applied layer by layer
        # After this call, every parameter's .grad attribute is populated
        loss.backward()
        
        # ---- Step 5: Optimizer step ----
        # Update all parameters: param = param - lr * gradient
        # (Adam actually uses a more sophisticated update rule with momentum)
        optimizer.step()
        
        # ---- Track metrics ----
        epoch_loss += loss.item()  # .item() converts 0-dim tensor to Python float
        
        # Compute accuracy: which class has highest logit?
        _, predicted = torch.max(logits, dim=1)  # Get index of max value
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    # ---- Epoch-level metrics ----
    avg_loss = epoch_loss / len(train_loader)  # Average over all batches
    accuracy = 100.0 * correct / total
    current_lr = optimizer.param_groups[0]['lr']
    
    train_losses.append(avg_loss)
    lr_history.append(current_lr)
    
    # ---- Learning rate scheduler step ----
    # Tell scheduler the current loss; it decides whether to reduce LR
    scheduler.step(avg_loss)
    
    # Print progress
    print(f"{epoch+1:<8} {avg_loss:<12.4f} {accuracy:<12.1f}% {current_lr:<12.6f}")

print("\nTraining complete!")
print(f"Final loss: {train_losses[-1]:.4f}")
print(f"Final accuracy: {accuracy:.1f}%")

### Visualizing Training Progress

**WHAT**: Plot the training loss curve over epochs.

**WHY**: Visual inspection tells us:
- Is the model learning? (loss should decrease)
- Is it converging? (loss should plateau eventually)
- Is it overfitting? (would show as train loss going to 0 while test loss increases)

**HOW it connects**: If the loss is not decreasing, something is wrong
(learning rate too high/low, bug in data pipeline, etc.)

In [ ]:
# Plot training loss and learning rate
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss
ax1.plot(range(1, NUM_EPOCHS + 1), train_losses, 'b-o', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss Over Epochs')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, NUM_EPOCHS)

# Plot 2: Learning Rate
ax2.plot(range(1, NUM_EPOCHS + 1), lr_history, 'r-o', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, NUM_EPOCHS)
ax2.set_yscale('log')  # Log scale for LR

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Loss should decrease over epochs (model is learning)")
print("- Learning rate may decrease if scheduler detects plateau")
print("- If loss is not decreasing: try higher learning rate or check data")

---
## Section 6: Evaluation - Cover Song Identification

### How Cover Song Retrieval Works

Now that training is complete, we use the model differently from how we trained it:

**During training**: Input CQT -> Model -> Classification logits (which song is this?)

**During evaluation**: Input CQT -> Model -> **300-dim embedding** (extract from fc0)

The evaluation pipeline:
1. Extract 300-dim embeddings for ALL songs in the test set
2. L2-normalize each embedding (so cosine similarity = dot product)
3. For each query song, compute similarity to all other songs
4. Rank by similarity and check: are the true covers ranked near the top?
5. Compute Mean Average Precision (MAP) as the overall metric

### What is L2 Normalization?

L2 normalization divides each vector by its length (magnitude):
`v_normalized = v / ||v||`

After normalization, all vectors have unit length (||v|| = 1).
This means cosine similarity = dot product:
`cos(a, b) = (a . b) / (||a|| * ||b||)` but if ||a||=||b||=1, then `cos(a, b) = a . b`

**WHY normalize?**: Removes the effect of embedding magnitude.
We only care about DIRECTION (what patterns), not magnitude (how strongly activated).

### Extracting Embeddings

**WHAT**: Pass all test samples through the trained model and collect
the 300-dimensional embedding vectors.

**WHY**: These embeddings are the compact representations of each song.
Similar songs (covers) should have similar embeddings.

**HOW it connects**: Once we have embeddings, we compute a similarity matrix
and evaluate retrieval performance.

In [ ]:
def extract_embeddings(model, data_loader, device):
    """
    Extract 300-dim embeddings for all samples in the data loader.
    
    Parameters:
    -----------
    model : CQTNet
        The trained model
    data_loader : DataLoader
        DataLoader containing samples to embed
    device : torch.device
        CPU or CUDA device
    
    Returns:
    --------
    all_embeddings : np.ndarray
        Shape: (num_samples, 300)
    all_labels : np.ndarray
        Shape: (num_samples,)
    """
    # Set model to evaluation mode
    # This changes BatchNorm behavior: uses running statistics instead of batch stats
    # Also disables dropout (if any)
    model.eval()
    
    all_embeddings = []  # Will collect embedding tensors
    all_labels = []      # Will collect labels
    
    # torch.no_grad(): disable gradient computation
    # WHY: during evaluation we don't need gradients (no backprop)
    # This saves memory and computation (~50% speedup)
    with torch.no_grad():
        for data, labels in data_loader:
            # data shape: (batch_size, 1, 84, out_length)
            data = data.to(device)
            
            # Forward pass - get both outputs
            logits, embedding = model(data)
            # embedding shape: (batch_size, 300)
            
            # Move to CPU and convert to numpy
            all_embeddings.append(embedding.cpu().numpy())
            all_labels.append(labels.numpy())
    
    # Concatenate all batches into single arrays
    all_embeddings = np.concatenate(all_embeddings, axis=0)  # (N, 300)
    all_labels = np.concatenate(all_labels, axis=0)          # (N,)
    
    return all_embeddings, all_labels


# Extract embeddings for ALL data (train + test) for visualization
# In practice, you would only extract for the query/gallery sets

# Create a combined loader for all files
all_dataset = CQTDataset(file_list, out_length=OUT_LENGTH, is_training=False)
all_loader = DataLoader(all_dataset, batch_size=BATCH_SIZE, shuffle=False)

embeddings, labels = extract_embeddings(model, all_loader, device)

print(f"Extracted embeddings:")
print(f"  Shape: {embeddings.shape} (num_samples x embedding_dim)")
print(f"  Labels: {labels.shape}")
print(f"  Unique classes: {len(np.unique(labels))}")

### L2 Normalization of Embeddings

**WHAT**: Normalize each 300-dim embedding to have unit length.

**WHY**: After normalization:
- All embeddings lie on a unit hypersphere (surface of a 300-dim ball)
- Cosine similarity = simple dot product (computationally cheap)
- Removes the effect of embedding magnitude
- Only the DIRECTION matters (which patterns are encoded)

**HOW it connects**: The normalized embeddings are used to compute the similarity matrix.

In [ ]:
# L2 normalization: divide each vector by its L2 norm (length)
# Formula: v_normalized = v / ||v||_2
# where ||v||_2 = sqrt(sum(v_i^2))

# Compute L2 norm for each embedding
# keepdims=True so we can divide (N, 300) by (N, 1)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)  # Shape: (N, 1)

# Avoid division by zero (shouldn't happen, but safety first)
norms = np.maximum(norms, 1e-8)

# Normalize
embeddings_normalized = embeddings / norms  # Shape: (N, 300)

# Verify: all norms should now be 1.0
new_norms = np.linalg.norm(embeddings_normalized, axis=1)
print(f"After L2 normalization:")
print(f"  Embedding shape: {embeddings_normalized.shape}")
print(f"  Norm range: [{new_norms.min():.6f}, {new_norms.max():.6f}] (should be ~1.0)")
print(f"  Mean norm: {new_norms.mean():.6f}")

### Computing the Similarity Matrix

**WHAT**: Compute pairwise cosine similarity between all pairs of embeddings.

**WHY**: This matrix tells us how similar each pair of songs is, according to the model.
For good performance:
- Songs from the same class (covers) should have HIGH similarity (~1.0)
- Songs from different classes should have LOW similarity (~0.0)

**HOW it connects**: We use this matrix to rank songs and compute MAP.

Since embeddings are L2-normalized, cosine similarity = dot product:
`similarity_matrix = embeddings @ embeddings.T`

In [ ]:
# Compute pairwise cosine similarity matrix
# Since embeddings are L2-normalized, cosine sim = dot product
# Matrix multiplication: (N, 300) @ (300, N) = (N, N)
similarity_matrix = embeddings_normalized @ embeddings_normalized.T

print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"  - Entry [i,j] = cosine similarity between song i and song j")
print(f"  - Diagonal should be 1.0 (song is identical to itself)")
print(f"  - Range: [{similarity_matrix.min():.3f}, {similarity_matrix.max():.3f}]")
print(f"  - Diagonal values: {similarity_matrix.diagonal()[:5]}... (should all be ~1.0)")

### Visualizing the Similarity Matrix

**WHAT**: Plot the similarity matrix as a heatmap.

**WHY**: Visual inspection quickly reveals:
- Block-diagonal structure = model learned to cluster covers together (GOOD)
- Uniform/random pattern = model has not learned useful features (BAD)

**Expected for demo**: Should show block-diagonal pattern because covers of
the same synthetic song share base patterns.

In [ ]:
# Plot the similarity matrix as a heatmap
fig, ax = plt.subplots(1, 1, figsize=(8, 7))

# Create the heatmap
im = ax.imshow(similarity_matrix, cmap='hot', vmin=-0.5, vmax=1.0)

# Add colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Cosine Similarity')

# Label axes
ax.set_xlabel('Song Index')
ax.set_ylabel('Song Index')
ax.set_title('Pairwise Cosine Similarity Matrix\n(bright blocks on diagonal = covers correctly grouped)')

# Add grid lines to show class boundaries
if DEMO_MODE:
    covers_per_song = 3
    for i in range(1, NUM_CLASSES):
        ax.axhline(y=i*covers_per_song - 0.5, color='cyan', linewidth=0.5, alpha=0.5)
        ax.axvline(x=i*covers_per_song - 0.5, color='cyan', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Bright blocks along the diagonal = covers of same song are similar (GOOD)")
print("- Dark off-diagonal regions = different songs are dissimilar (GOOD)")
print("- If everything is uniformly bright/dark = model has not learned well")

### Mean Average Precision (MAP) Calculation

**WHAT**: Compute MAP, the standard metric for cover song identification.

**WHY**: MAP answers: "When we search for covers of a song, how well do the
true covers rank among all candidates?"

**How MAP works**:
1. For each query song, rank all other songs by similarity (highest first)
2. Compute Average Precision (AP) for this query:
   - Go through the ranked list
   - Each time a true cover is found, compute precision@k (fraction of covers in top-k)
   - AP = mean of these precision values
3. MAP = mean of AP over all queries

**Example**:
- Song has 2 covers. Ranking: [cover, non-cover, cover, non-cover, ...]
- Precision@1 = 1/1 = 1.0 (first result is correct)
- Precision@3 = 2/3 = 0.67 (second cover found at position 3)
- AP = (1.0 + 0.67) / 2 = 0.83

**Perfect MAP = 1.0** (all covers ranked at the very top)

In [ ]:
def compute_map(similarity_matrix, labels):
    """
    Compute Mean Average Precision for cover song retrieval.
    
    For each song (query), rank all other songs by similarity.
    Check how well the true covers rank.
    
    Parameters:
    -----------
    similarity_matrix : np.ndarray
        Shape: (N, N). Pairwise similarity scores.
    labels : np.ndarray
        Shape: (N,). Class label for each song.
    
    Returns:
    --------
    mean_ap : float
        Mean Average Precision (0 to 1, higher = better)
    per_query_ap : list
        AP for each query
    """
    N = len(labels)
    per_query_ap = []  # Store AP for each query
    
    for i in range(N):
        # Current query song
        query_label = labels[i]
        
        # Get similarities to all OTHER songs (exclude self)
        similarities = similarity_matrix[i].copy()
        similarities[i] = -1.0  # Exclude self-match
        
        # Find which songs are true covers (same class, excluding self)
        # These are the "relevant" items we want to find
        relevant_mask = (labels == query_label)
        relevant_mask[i] = False  # Exclude self
        
        num_relevant = relevant_mask.sum()
        if num_relevant == 0:
            continue  # Skip if no covers exist for this song
        
        # Sort all songs by similarity (descending = most similar first)
        ranked_indices = np.argsort(-similarities)  # Negative for descending
        
        # Compute Average Precision
        # Go through ranked list, accumulate precision at each relevant hit
        hits = 0  # Number of relevant items found so far
        sum_precision = 0.0
        
        for rank, idx in enumerate(ranked_indices):
            if idx == i:  # Skip self
                continue
            if relevant_mask[idx]:  # Found a true cover!
                hits += 1
                # Precision at this rank = hits / (rank + 1)
                precision_at_k = hits / (rank + 1)
                sum_precision += precision_at_k
            
            if hits == num_relevant:  # Found all relevant items
                break
        
        # AP = sum of precisions / number of relevant items
        ap = sum_precision / num_relevant
        per_query_ap.append(ap)
    
    # MAP = mean of all per-query APs
    mean_ap = np.mean(per_query_ap)
    
    return mean_ap, per_query_ap


# Compute MAP on our embeddings
mean_ap, per_query_ap = compute_map(similarity_matrix, labels)

print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Mean Average Precision (MAP): {mean_ap:.4f}")
print(f"  - Perfect score: 1.0000")
print(f"  - Random baseline: ~{1.0/NUM_CLASSES:.4f}")
print(f"\nPer-song AP values:")
for i, ap in enumerate(per_query_ap[:10]):  # Show first 10
    print(f"  Song {labels[i]}: AP = {ap:.4f}")
if len(per_query_ap) > 10:
    print(f"  ... ({len(per_query_ap) - 10} more)")

### Faculty Q&A: Evaluation

**Q: Why use MAP instead of simple accuracy?**

A: Accuracy measures classification (is this song X? yes/no).
But cover song identification is a RETRIEVAL task: given a query,
rank all candidates and return the best matches. MAP measures ranking
quality, which is exactly what matters in practice.

**Q: What is a good MAP score for Indian music?**

A: It depends on the dataset size and difficulty:
- MAP > 0.80: Excellent (most covers ranked very high)
- MAP 0.50-0.80: Good (covers are generally ranked near the top)
- MAP 0.20-0.50: Moderate (model is learning but struggles with some songs)
- MAP < 0.20: Poor (barely better than random for a small dataset)

For a small Indian dataset (245 songs), MAP > 0.5 is a reasonable target
for a from-scratch model. Fine-tuning from pretrained weights might reach > 0.7.

**Q: Why extract embeddings instead of using classification output?**

A: The classification head (fc1) only works for songs seen during training.
It cannot handle new songs. The embedding from fc0 is a general-purpose
representation that works for ANY song, including ones never seen during
training. This generalization is what makes it useful for real applications.

---
## Section 7: How to Use With Your Real Indian Music Data

### Step-by-Step Instructions

Now that you understand the complete pipeline, here is how to train on your
actual Indian music dataset.

#### Step 1: Organize Your MP3 Files

Create the following folder structure:
```
dataset/
  song_001_raga_bhairavi/
    original.mp3
    cover_artist_A.mp3
    cover_artist_B.mp3
  song_002_tum_hi_ho/
    original.mp3
    cover_live_version.mp3
    cover_acoustic.mp3
  ...
```

Rules:
- Each folder = one original song (one class)
- All MP3 files within a folder are different versions/covers of that song
- Folder names can be anything (they are sorted alphabetically for class IDs)
- You need AT LEAST 2 files per folder (otherwise no cover to learn from)
- More covers per song = better training signal

#### Step 2: Run CQT Generation

Set `DEMO_MODE = False` and `DATA_DIR` to your folder path in Section 1,
then run all cells. The CQT features will be automatically computed and saved.

#### Step 3: Adjust num_classes

Set `NUM_CLASSES` to the number of unique songs (folders) in your dataset.
For example, if you have 245 original songs: `NUM_CLASSES = 245`

#### Step 4: Recommended Hyperparameters for Small Datasets

| Parameter | Small (< 50 songs) | Medium (50-200 songs) | Large (200+ songs) |
|-----------|-------------------|----------------------|-------------------|
| Batch size | 8-16 | 16-32 | 32-64 |
| Learning rate | 5e-4 | 1e-3 | 1e-3 |
| Weight decay | 1e-3 | 1e-4 | 1e-4 |
| Epochs | 50-100 | 100-200 | 200-500 |
| out_length | 200 | 200-300 | 200-400 |

For small datasets, increase weight_decay to prevent overfitting.

#### Step 5: Multi-Size Training (Advanced)

The original CQTNet trains with out_length = 200, 300, and 400 in rotation.
This makes the model robust to different temporal scales.
To implement: cycle through different OUT_LENGTH values across epochs.

### Saving and Loading the Trained Model

**WHAT**: Save the trained model weights to disk and demonstrate how to reload them.

**WHY**: Training takes time. Saving allows you to:
- Resume training later
- Deploy the model without retraining
- Share the trained model

**HOW it connects**: In a real workflow, you train once, save,
then load the model whenever you need to identify cover songs.

In [ ]:
# ============================================================
# SAVING THE MODEL
# ============================================================

# Save model weights (state_dict)
# This saves ONLY the learned parameters, not the model architecture
MODEL_SAVE_PATH = './cqtnet_indian_music.pth'

torch.save({
    'model_state_dict': model.state_dict(),  # All learned weights
    'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state (for resume)
    'num_classes': NUM_CLASSES,  # Architecture info needed to rebuild model
    'epoch': NUM_EPOCHS,  # Training progress
    'train_loss': train_losses[-1],  # Final loss
}, MODEL_SAVE_PATH)

print(f"Model saved to: {MODEL_SAVE_PATH}")
print(f"File size: {os.path.getsize(MODEL_SAVE_PATH) / 1024 / 1024:.1f} MB")

### Loading a Saved Model

**WHAT**: Demonstrate how to load a previously saved model.

**WHY**: You need this to:
- Use the trained model for inference (cover song identification)
- Resume training from a checkpoint
- Deploy the model in production

**HOW it connects**: After loading, the model is ready to extract embeddings
for any new audio file.

In [ ]:
# ============================================================
# LOADING THE MODEL
# ============================================================

# Step 1: Load the checkpoint
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)

# Step 2: Create a new model with the same architecture
loaded_model = CQTNet(num_classes=checkpoint['num_classes']).to(device)

# Step 3: Load the saved weights into the model
loaded_model.load_state_dict(checkpoint['model_state_dict'])

# Step 4: Set to evaluation mode
loaded_model.eval()

print(f"Model loaded from: {MODEL_SAVE_PATH}")
print(f"  Classes: {checkpoint['num_classes']}")
print(f"  Trained for: {checkpoint['epoch']} epochs")
print(f"  Final loss: {checkpoint['train_loss']:.4f}")
print(f"\nModel is ready for inference!")

### Complete Inference Pipeline: Identifying a Cover Song

**WHAT**: Show the full end-to-end pipeline for identifying whether a new song
is a cover of any song in the database.

**WHY**: This is the practical application -- given a new audio file,
find its covers in the database.

**HOW it connects**: This demonstrates the real-world use case of the entire system.

In [ ]:
def identify_cover_song(query_audio_path, model, database_embeddings, 
                         database_labels, song_names=None, sr=22050, 
                         mean_size=20, out_length=200, device='cpu'):
    """
    Full pipeline to identify if a query audio is a cover of any song in the database.
    
    Parameters:
    -----------
    query_audio_path : str
        Path to the query MP3 file
    model : CQTNet
        Trained model (in eval mode)
    database_embeddings : np.ndarray
        L2-normalized embeddings of all songs in database. Shape: (N, 300)
    database_labels : np.ndarray
        Class labels of database songs. Shape: (N,)
    song_names : list, optional
        Human-readable names for each class
    sr : int
        Sample rate for audio loading
    mean_size : int
        Temporal pooling window
    out_length : int
        Fixed temporal length for model input
    device : str or torch.device
        Compute device
    
    Returns:
    --------
    results : list of tuples
        Sorted list of (similarity_score, class_label, song_name)
    """
    # Step 1: Compute CQT features for the query
    cqt_features = compute_cqt_features(query_audio_path, sr=sr, mean_size=mean_size)
    # cqt_features shape: (84, T)
    
    # Step 2: Preprocess (same as Dataset.__getitem__)
    cqt = cqt_features.T  # (T, 84)
    mean, std = cqt.mean(), cqt.std()
    if std > 0:
        cqt = (cqt - mean) / std
    
    # Crop or pad to out_length
    T = cqt.shape[0]
    if T >= out_length:
        start = (T - out_length) // 2  # Center crop
        cqt = cqt[start:start + out_length, :]
    else:
        padding = np.zeros((out_length - T, 84))
        cqt = np.concatenate([cqt, padding], axis=0)
    
    # Reshape to (1, 1, 84, out_length)
    cqt = cqt.T[np.newaxis, np.newaxis, :, :]  # (1, 1, 84, out_length)
    
    # Step 3: Extract embedding
    model.eval()
    with torch.no_grad():
        tensor = torch.FloatTensor(cqt).to(device)
        _, embedding = model(tensor)
        query_emb = embedding.cpu().numpy()  # (1, 300)
    
    # Step 4: L2 normalize
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    
    # Step 5: Compute similarity to all database songs
    similarities = (query_emb @ database_embeddings.T).flatten()  # (N,)
    
    # Step 6: Rank by similarity
    ranked_indices = np.argsort(-similarities)  # Descending
    
    results = []
    for idx in ranked_indices[:10]:  # Top 10 matches
        name = song_names[database_labels[idx]] if song_names else f"Song {database_labels[idx]}"
        results.append((similarities[idx], database_labels[idx], name))
    
    return results


print("Function defined: identify_cover_song()")
print("\nUsage:")
print("  results = identify_cover_song('query.mp3', model, db_embeddings, db_labels)")
print("  for sim, label, name in results:")
print("      print(f'{name}: similarity = {sim:.3f}')")

### Demo: Simulating Cover Song Identification

**WHAT**: Since we may not have real MP3 files, we simulate the retrieval
process using our synthetic test data.

**WHY**: Demonstrates that the pipeline works end-to-end.
Each test sample should be most similar to other samples from the same class.

In [ ]:
# Simulate retrieval using our test embeddings
print("Cover Song Identification Results (Simulated)")
print("=" * 50)

# Use our already-computed normalized embeddings as the "database"
# Pick a few query samples and find their nearest neighbors

num_queries = min(5, len(labels))  # Show 5 queries

for query_idx in range(0, len(labels), max(1, len(labels) // num_queries)):
    if query_idx >= len(labels):
        break
    
    query_label = labels[query_idx]
    
    # Get similarities to all other songs
    sims = similarity_matrix[query_idx].copy()
    sims[query_idx] = -1  # Exclude self
    
    # Get top 3 matches
    top_indices = np.argsort(-sims)[:3]
    
    print(f"\nQuery: Sample {query_idx} (Class/Song: {query_label})")
    print(f"  Top matches:")
    for rank, idx in enumerate(top_indices):
        match_label = labels[idx]
        is_cover = "COVER" if match_label == query_label else "different song"
        print(f"    Rank {rank+1}: Sample {idx} (Class: {match_label}) "
              f"sim={sims[idx]:.3f} [{is_cover}]")

print(f"\n{'='*50}")
print(f"Overall MAP: {mean_ap:.4f}")
print(f"\nNote: With real Indian music and more training epochs,")
print(f"the model should achieve significantly better MAP scores.")

### Summary and Next Steps

#### What We Built

1. **CQT Feature Extraction**: Audio -> 84-bin CQT spectrogram -> temporal pooling
2. **Dataset Pipeline**: Load .npy files, normalize, random crop, batch
3. **CQTNet Model**: 10-layer CNN that produces 300-dim embeddings
4. **Training**: Cross-entropy classification with Adam optimizer
5. **Evaluation**: Extract embeddings, normalize, compute similarity, measure MAP

#### Key Takeaways for Faculty Explanation

- CQT is chosen because its logarithmic frequency scale matches musical structure
- We train as classification but USE embeddings for retrieval
- The model is trained FROM SCRATCH (no Western pretrained weights)
- AdaptiveMaxPool enables variable-length input handling
- L2 normalization + dot product = cosine similarity for efficient retrieval

#### Potential Improvements

1. **Data augmentation**: pitch shifting, time stretching, adding noise
2. **Triplet loss**: mine hard negatives for better embedding separation
3. **Multi-size training**: cycle through out_length = 200, 300, 400
4. **Larger dataset**: more songs and covers improve generalization
5. **Transfer learning**: start from Western pretrained weights, fine-tune on Indian music

#### Complete Code Summary

To train on your own data:
```python
DEMO_MODE = False
DATA_DIR = '/path/to/your/indian_music_dataset/'
NUM_CLASSES = 245  # Your number of songs
NUM_EPOCHS = 100
BATCH_SIZE = 32
```
Then run all cells from top to bottom.